# MetalDAM → PSCL Finetuning Walkthrough

This notebook guides you through the complete process of adapting MetalDAM's preprocessed
SEM micrograph patches to finetune the **PSCL** (Patch-Sampled Contrastive Learning) segmentation model.

**What you will learn:**
- What MetalDAM produces and what PSCL expects
- How the `Pscl_fintune` project bridges that gap
- How self-supervised contrastive pretraining works (Stage 1)
- How supervised finetuning works (Stage 2)
- How to read and visualise results

**Requirements:**  `conda activate prep` | GPU with CUDA | MetalDAM pipeline already run

---
## 0 · Project folder overview

The `Pscl_fintune/` directory is a **standalone adapter project**. It imports from
`PSCL/PSCL/` at runtime but never modifies it.

```
Pscl_fintune/
│
├── data_metaldam.py      ← FORMAT BRIDGE
│     Three PyTorch Dataset classes that convert MetalDAM patches
│     (256×256 grayscale, 5 classes) into what PSCL expects
│     (376×376 RGB, 4 classes, one-hot encoded).
│
├── finetune.py           ← TRAINING LOOPS
│     SelfSupervised_MetalDAM()  — Stage 1: contrastive pretraining
│     Finetune_MetalDAM()        — Stage 2: supervised segmentation
│     load_moco()                — loads Stage 1 encoder into Stage 2 model
│
├── config_metaldam.py    ← HYPERPARAMETERS
│     MetalDAMConfig class — all parameters in one place
│     run(method='self'|'fine') — one-line entry point
│
├── train_metaldam.py     ← CLI ENTRY POINT
│     python train_metaldam.py --stage self   # pretrain
│     python train_metaldam.py --stage fine   # finetune
│     python train_metaldam.py                # both
│
├── INSTRUCTIONS.md       ← step-by-step run guide
├── FINETUNING_GUIDE.md   ← flowchart and concept overview
└── .claude/rules/        ← coding invariants for Claude Code
```

**External dependency (read-only):**
```
PSCL/PSCL/
  model.py      ← UNet + MoCo_DenseModel architectures
  networks.py   ← projection head (Denseproj_UNET_MLP)
  data.py       ← get_one_hot() utility
  utils.py      ← Averagvalue, Logger
```

In [ ]:
import os, sys, re, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import cv2
import torch

# ── paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR     = Path(os.path.abspath('')).resolve()   # Pscl_fintune/
PSCL_SRC         = str(NOTEBOOK_DIR.parent / 'PSCL' / 'PSCL')
METALDAM_PATCHES = str(NOTEBOOK_DIR.parent / 'MetalDam' / 'data' / 'patches')

if PSCL_SRC not in sys.path:
    sys.path.insert(0, PSCL_SRC)

os.chdir(str(NOTEBOOK_DIR))          # checkpoints are written relative to here

print(f'Notebook dir : {NOTEBOOK_DIR}')
print(f'PSCL source  : {PSCL_SRC}')
print(f'MetalDAM data: {METALDAM_PATCHES}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch version: {torch.__version__}')

---
## 1 · MetalDAM dataset — what it contains

MetalDAM preprocessed 42 raw SEM micrographs of steel into **1,148 labelled patches**
through an 8-node pipeline. The patches are split at the **parent-image level** (no leakage):

| Split | Patches | Parent images |
|-------|---------|---------------|
| train | 788 | 29 |
| val | 168 | 7 |
| test | 192 | 6 |

Each patch is stored as:
- **Image** — 256×256 grayscale uint8 PNG in `images_norm/` (z-score normalised)
- **Mask** — 256×256 uint8 PNG in `masks/` with class indices:

| Index | Phase | Colour in label |
|-------|-------|-----------------|
| 0 | Background / Defect | Magenta |
| 1 | Austenite | Green |
| 2 | Matrix | Purple |
| 3 | Martensite-Austenite (MA) | Yellow |
| 4 | Precipitate | Red |
| 255 | Ignore (unmatched pixels) | — |

In [ ]:
# ── Visualise sample patches and their masks ─────────────────────────────────

# MetalDAM original class colours (for display)
METALDAM_COLORS = {
    0:   (255,   0, 255),   # Background  — magenta
    1:   ( 43, 255,   0),   # Austenite   — green
    2:   (128,   0, 255),   # Matrix      — purple
    3:   (255, 255,   0),   # MA          — yellow
    4:   (255,   0,   0),   # Precipitate — red
    255: (  0,   0,   0),   # Ignore      — black
}
PHASE_NAMES = ['Background', 'Austenite', 'Matrix', 'MA', 'Precipitate']

def colorise_mask(mask):
    """Convert a MetalDAM class-index mask to an RGB image for display."""
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for idx, color in METALDAM_COLORS.items():
        rgb[mask == idx] = color
    return rgb

# pick 6 random training patches
train_img_dir  = os.path.join(METALDAM_PATCHES, 'train', 'images_norm')
train_mask_dir = os.path.join(METALDAM_PATCHES, 'train', 'masks')
all_fnames     = sorted(f for f in os.listdir(train_img_dir) if f.endswith('.png'))
sample_fnames  = random.sample(all_fnames, 6)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('MetalDAM training patches (top: image, bottom: class mask)', fontsize=13)

for col, fname in enumerate(sample_fnames):
    img  = cv2.imread(os.path.join(train_img_dir,  fname), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(os.path.join(train_mask_dir, fname), cv2.IMREAD_GRAYSCALE)
    axes[0, col].imshow(img, cmap='gray')
    axes[0, col].set_title(fname[:20], fontsize=7)
    axes[0, col].axis('off')
    axes[1, col].imshow(colorise_mask(mask))
    axes[1, col].axis('off')

# legend
legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                  for n, c in zip(PHASE_NAMES, list(METALDAM_COLORS.values())[:5])]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=9, frameon=False)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# ── Class distribution across the training split ──────────────────────────────
import json

# Read class histograms from metadata.csv
import pandas as pd
meta = pd.read_csv(str(NOTEBOOK_DIR.parent / 'MetalDam' / 'metadata.csv'))
train_meta = meta[(meta['split'] == 'train') & meta['patch_y'].notna()].copy()

totals = {str(i): 0 for i in range(5)}
for hist_str in train_meta['class_histogram'].dropna():
    hist = json.loads(hist_str)
    for k, v in hist.items():
        if k in totals:
            totals[k] += v

labels  = [f'{PHASE_NAMES[i]}\n(class {i})' for i in range(5)]
counts  = [totals[str(i)] for i in range(5)]
colors  = [np.array(METALDAM_COLORS[i])/255 for i in range(5)]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Total pixels (training split)')
ax.set_title('Class distribution in MetalDAM training patches')
for bar, count in zip(bars, counts):
    pct = 100 * count / sum(counts)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('Note: MA and Precipitate are rare → we upweight them in the loss (weight=[1,1,5,5])')

---
## 2 · Format mismatch and class remapping

PSCL was designed for an **aluminum** dataset with different properties:

| Property | MetalDAM output | PSCL expects | Solution |
|----------|----------------|--------------|----------|
| Image size | 256 × 256 | 376 × 376 | `cv2.resize` INTER_LINEAR |
| Channels | 1 (grayscale) | 3 (RGB) | repeat channel ×3 |
| Normalisation | z-score baked in | ImageNet mean/std | re-normalise |
| Classes | 5 (indices 0–4) + 255 ignore | 4 (one-hot) | remap: Background→ignore, shift 1–4 → 0–3 |
| Mask encoding | uint8 integer index | (4, H, W) float32 | `get_one_hot` + permute |

### Why remap Background to ignore?
PSCL's UNet has exactly **4 output channels** (hardcoded). MetalDAM has 5 classes.
Background/Defect pixels (class 0) are not a meaningful microstructure phase,
so we treat them as "ignore" (label 255). The loss never penalises predictions
on these pixels. This lets the 4 real phases (Austenite, Matrix, MA, Precipitate)
map cleanly to PSCL indices 0–3 without any model architecture change.

In [ ]:
# ── Class remapping visualisation ────────────────────────────────────────────

PSCL_COLORS = {
    0:   ( 43, 255,   0),   # Austenite   (was MetalDAM 1)
    1:   (128,   0, 255),   # Matrix      (was MetalDAM 2)
    2:   (255, 255,   0),   # MA          (was MetalDAM 3)
    3:   (255,   0,   0),   # Precipitate (was MetalDAM 4)
    255: ( 30,  30,  30),   # Ignore      (was MetalDAM 0 Background)
}
PSCL_NAMES = ['Austenite (0)', 'Matrix (1)', 'MA (2)', 'Precipitate (3)', 'Ignore (255)']

_REMAP = np.array([255, 0, 1, 2, 3], dtype=np.uint8)  # index i → PSCL class

def colorise_pscl_mask(mask_remapped):
    rgb = np.zeros((*mask_remapped.shape, 3), dtype=np.uint8)
    for idx, color in PSCL_COLORS.items():
        rgb[mask_remapped == idx] = color
    return rgb

# pick a patch that has all 5 classes
selected = None
for fname in all_fnames:
    m = cv2.imread(os.path.join(train_mask_dir, fname), cv2.IMREAD_GRAYSCALE)
    if all(np.any(m == c) for c in [0, 1, 2, 3, 4]):
        selected = fname; break
if selected is None:
    selected = all_fnames[0]

img_orig  = cv2.imread(os.path.join(train_img_dir, selected), cv2.IMREAD_GRAYSCALE)
mask_orig = cv2.imread(os.path.join(train_mask_dir, selected), cv2.IMREAD_GRAYSCALE)
mask_remap = _REMAP[mask_orig]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle(f'Class remapping — {selected[:35]}', fontsize=11)

axes[0].imshow(img_orig, cmap='gray')
axes[0].set_title('SEM image (grayscale)')
axes[0].axis('off')

axes[1].imshow(colorise_mask(mask_orig))
axes[1].set_title('MetalDAM mask (5 classes)')
axes[1].axis('off')
leg1 = [mpatches.Patch(color=np.array(METALDAM_COLORS[i])/255, label=PHASE_NAMES[i])
        for i in range(5)]
axes[1].legend(handles=leg1, loc='lower right', fontsize=7, framealpha=0.8)

axes[2].imshow(colorise_pscl_mask(mask_remap))
axes[2].set_title('PSCL mask (4 classes + ignore)')
axes[2].axis('off')
leg2 = [mpatches.Patch(color=np.array(PSCL_COLORS[k])/255, label=n)
        for k, n in zip([0,1,2,3,255], PSCL_NAMES)]
axes[2].legend(handles=leg2, loc='lower right', fontsize=7, framealpha=0.8)

# annotation arrows
fig.text(0.49, 0.5, '→\nremap\nBackground\n→ ignore\nshift 1–4\n→ 0–3',
         ha='center', va='center', fontsize=8, color='dimgray')
plt.tight_layout()
plt.show()

---
## 3 · Background: What is PSCL?

**PSCL = Patch-Sampled Contrastive Learning**

It is a self-supervised pretraining method designed specifically for **dense prediction
(segmentation)** on metallographic images, where labelled data is scarce.

### The core idea: contrastive learning

In contrastive learning, the model learns by answering the question:
*"Are these two image patches from the same category or not?"*

```
Anchor patch  ─── same class? ──→  Positive (pull closer in feature space)
              ─── different?  ──→  Negative (push apart)
```

The model is trained to produce **feature embeddings** where same-class patches
cluster together. No class labels are needed for most images — the label from a
*small* annotated set guides *which* patches are sampled as positives/negatives.

### PSCL's innovation: supervised patch sampling

Standard contrastive learning uses random augmentations to define positives.
PSCL additionally uses a **small labelled sample** to pick patches that are
semantically similar across different images — making the contrastive signal
much stronger for segmentation.

### Two-stage pipeline

```
Stage 1 — Self-supervised pretraining
  Input : All training patches (unlabeled) + a small labeled subset
  Output: A pretrained UNet encoder with domain-adapted features
  Time  : ~200 epochs

Stage 2 — Supervised finetuning
  Input : Pretrained encoder + all labeled training patches
  Output: A segmentation model (4-class UNet)
  Time  : ~50 epochs
```

### Model architecture

```
Input [B, 3, 376, 376]
      │
   UNet Encoder (4 downsampling stages, channels: 64→128→256→512)
      │  ←── Stage 1: contrastive projection head attached here
      │  ←── Stage 2: segmentation decoder attached here
      │
   UNet Decoder (4 upsampling stages with skip connections)
      │
Output [B, 4, 376, 376]  ← 4 class logits
```

---
## 4 · The data adapter — `data_metaldam.py`

This file contains three PyTorch `Dataset` classes that wrap MetalDAM patches and
deliver them in the exact format PSCL's training loops expect.

### `MetalDAMDataset` — for supervised finetuning
Returns `(image [3,376,376] float32,  gt_onehot [4,376,376] float32)` for train.  
Returns `(image, gt_onehot, filename)` for val/test.

### `MetalDAMMoCoDataset` — unlabeled pool for Stage 1
Generates **two differently augmented views** of each image (query + key).  
Returns `(cat[view_q, view_k] [6,376,376], id_str, rot_k, flip, r1, r2, r3, r4)`.

### `MetalDAMMoCoDatasetSup` — labeled guidance for Stage 1
Same as above but also returns the one-hot mask (scaled ×255) concatenated as channel 7–10.  
Returns `(cat[view_q, view_k, gt×255] [10,376,376], id_str, rot_k, flip, r1, r2, r3, r4)`.

> **Why scale gt×255?**  
> PSCL's model uses these values to identify which pixels belong to each class when
> sampling contrastive patch pairs. Values of 0 or 255 make class membership binary
> and easy to threshold. Pixels remapped to ignore (255) are zeroed across all channels.

In [ ]:
# ── Inspect what MetalDAMDataset actually returns ────────────────────────────
from data_metaldam import MetalDAMDataset, MetalDAMMoCoDataset, MetalDAMMoCoDatasetSup

ds = MetalDAMDataset(METALDAM_PATCHES, split='train')
img, gt = ds[0]

print('=== MetalDAMDataset (train) ===')
print(f'  Dataset size      : {len(ds)} patches')
print(f'  image shape       : {tuple(img.shape)}   dtype: {img.dtype}')
print(f'  image value range : [{img.min():.3f}, {img.max():.3f}]')
print(f'  gt_onehot shape   : {tuple(gt.shape)}   dtype: {gt.dtype}')
print(f'  gt unique values  : {gt.unique().tolist()}')

moco_ds = MetalDAMMoCoDataset(METALDAM_PATCHES)
inp, id_str, rot_k, filp, r1, r2, r3, r4 = moco_ds[0]

print('\n=== MetalDAMMoCoDataset (unlabeled pool) ===')
print(f'  Dataset size      : {len(moco_ds)} patches')
print(f'  input shape       : {tuple(inp.shape)}  (6 = view_q[3] + view_k[3])')
print(f'  id_str            : {id_str!r}  (split("_")[0] = {id_str.split("_")[0]!r} → size-group)')

sup_ds = MetalDAMMoCoDatasetSup(METALDAM_PATCHES, sup_split='val')
inp_s, id_s, *_ = sup_ds[0]
print('\n=== MetalDAMMoCoDatasetSup (labeled guidance, val split) ===')
print(f'  Dataset size      : {len(sup_ds)} patches')
print(f'  input shape       : {tuple(inp_s.shape)}  (10 = view_q[3] + view_k[3] + gt×255[4])')

In [ ]:
# ── Visualise one item from MetalDAMDataset ───────────────────────────────────
# The image has been resized to 376×376, replicated to 3 channels, and normalised.
# To display it we undo the normalisation.

MEAN = np.array([0.49139968, 0.48215841, 0.44653091])
STD  = np.array([0.24703223, 0.24348513, 0.26158784])

def unnormalise(tensor):
    img = tensor.numpy().transpose(1, 2, 0)   # (C,H,W) → (H,W,C)
    img = img * STD + MEAN
    return np.clip(img, 0, 1)

img_t, gt_t = ds[42]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('MetalDAMDataset item — as seen by PSCL', fontsize=11)

axes[0].imshow(unnormalise(img_t))
axes[0].set_title('Image tensor → display\n(376×376, normalised to ImageNet stats)')
axes[0].axis('off')

# convert one-hot back to class index for display
gt_np    = gt_t.numpy()                      # (4, 376, 376)
gt_cls   = np.argmax(gt_np, axis=0)          # (376, 376) — class 0-3
# mark pixels where all channels == 0 as ignore
gt_cls[gt_np.sum(axis=0) == 0] = 255
axes[1].imshow(colorise_pscl_mask(gt_cls.astype(np.uint8)))
axes[1].set_title('gt_onehot → PSCL display\n(class 0–3 + ignore=255)')
axes[1].axis('off')
axes[1].legend(handles=leg2, loc='lower right', fontsize=7, framealpha=0.8)

# show one channel of the one-hot
axes[2].imshow(gt_np[0], cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('One-hot channel 0\n(Austenite: 1=present, 0=absent)')
axes[2].axis('off')
plt.colorbar(axes[2].images[0], ax=axes[2], shrink=0.8)

plt.tight_layout()
plt.show()

---
## 5 · Configuration — `config_metaldam.py`

`MetalDAMConfig` holds every hyperparameter the training functions need.
The key parameters and their rationale:

In [ ]:
import config_metaldam

# Build a config in 'self' mode and inspect key attributes
cfg = config_metaldam.MetalDAMConfig(
    method='self',
    tt='metaldam',
    data_dir=METALDAM_PATCHES,
    self_max_epoch=200,
)

print('Key MetalDAMConfig parameters:')
print(f'  data_dir          : {cfg.data_dir}')
print(f'  selfmode          : {cfg.selfmode!r}           ("moco" or "simclr")')
print(f'  self_max_epoch    : {cfg.self_max_epoch}        (pretraining epochs)')
print(f'  self_save_epoch   : {cfg.self_save_epoch}         (save checkpoint every N epochs)')
print(f'  temperature       : {cfg.temperature}        (contrastive softmax temperature)')
print(f'  moco_denseloss_ratio: {cfg.moco_denseloss_ratio}     (weight of patch-level vs global loss)')
print(f'  top_k             : {cfg.top_k}           (patches sampled per class)')
print(f'  fine_max_epoch    : {cfg.fine_max_epoch}         (finetuning epochs)')
print(f'  fineturn_lr_en    : {cfg.fineturn_lr_en}      (encoder LR — small, preserves pretraining)')
print(f'  fineturn_lr_de    : {cfg.fineturn_lr_de}       (decoder LR — large, trains from scratch)')
print(f'  weight (BCE)      : {cfg.weight}  (upweight rare phases MA+Precipitate)')
print(f'  dice_weight       : {cfg.dice_weight}  (equal Dice contribution)')
print(f'  HeadNrom          : {cfg.HeadNrom}   (projection head norm: element [1] controls BN)')
print(f'  tmp dir           : {cfg.tmp}')

---
## 6 · Stage 1 — Self-supervised pretraining

### What happens in each training step:

```
For each batch:

  1. Draw B patches from the unlabeled pool (train split)
     → apply two DIFFERENT random augmentations → view_q, view_k

  2. Draw 1 patch from the labeled guidance pool (val split)
     → its one-hot mask × 255 tells PSCL which pixels belong to which class

  3. Forward through TWO encoders:
       encoder_q (trained with gradients)
       encoder_k (momentum-updated: encoder_k ← m·encoder_k + (1-m)·encoder_q)

  4. Extract features at 4 scales (global + 3 spatial resolutions)

  5. For the labeled patch:
       sample top-K patches per class from the feature maps
       build positive/negative pairs guided by class membership

  6. Compute loss:
       global InfoNCE loss (image-level contrastive)
     + dense  InfoNCE loss (patch-level contrastive, weighted by moco_denseloss_ratio)

  7. Backprop through encoder_q only
     Update memory queue (FIFO bank of past key features)
```

### The memory queue

MoCo maintains a queue of `queue_size=504` past key embeddings.
Each new batch compares against ALL entries in the queue as negatives,
giving a large effective negative set without needing a huge batch size.

In [ ]:
# ── Stage 1: smoke-test run (2 epochs) ───────────────────────────────────────
# For a real run use self_max_epoch=200 in train_metaldam.py.
# Here we run 2 epochs just to verify the pipeline works end-to-end.

print('Running 2 pretraining epochs (smoke test) ...')
print('For full training use: python train_metaldam.py --stage self')
print('-' * 60)

config_metaldam.run(
    method='self',
    tt='metaldam',
    data_dir=METALDAM_PATCHES,
    self_max_epoch=2,
    self_save_epoch=1,    # save every epoch for this demo
    env='0',
)

In [ ]:
# ── Parse and plot the training loss curve ────────────────────────────────────
import glob

log_pattern = 'self_UNet_metaldam_Numf/f/log_self.txt'

epochs, total_loss, c_loss, dense_loss = [], [], [], []

if os.path.exists(log_pattern):
    with open(log_pattern) as f:
        for line in f:
            m = re.search(r'Epoch \[(\d+)/\d+\].*Loss ([\d.]+).*C ([\d.]+).*Dense ([\d.]+)', line)
            if m:
                epochs.append(int(m.group(1)))
                total_loss.append(float(m.group(2)))
                c_loss.append(float(m.group(3)))
                dense_loss.append(float(m.group(4)))

if epochs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Stage 1 — Pretraining loss curves', fontsize=12)

    axes[0].plot(epochs, total_loss, 'b-o', markersize=3, label='Total loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Total loss (global + dense)')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, c_loss,     'g-o', markersize=3, label='Global (InfoNCE)')
    axes[1].plot(epochs, dense_loss, 'r-o', markersize=3, label='Dense (patch-level)')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].set_title('Component losses')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(f'Logged {len(epochs)} epochs. Final total loss: {total_loss[-1]:.4f}')
else:
    print(f'Log not found at {log_pattern}')
    print('Run the pretraining cell above first, or run: python train_metaldam.py --stage self')

---
## 7 · Stage 2 — Supervised finetuning

### What happens:

```
1. Load the pretrained encoder_q weights from Stage 1 checkpoint
   (load_moco strips the 'encoder_q.' prefix; projection head weights are dropped)

2. Attach a fresh UNet decoder (4 output channels, randomly initialised)

3. Optimise with Adam — TWO learning rate groups:
     encoder layers (inc, down1–3):  lr = 1e-4  ← small: preserve pretrained features
     decoder + up layers (up1–3):    lr = 1e-3  ← large: train from scratch

4. Loss per batch:
     BCE with class weights [1, 1, 5, 5]  (upweight rare MA and Precipitate)
   + Dice loss
   = (1 - dice_bce_ratio)×BCE + dice_bce_ratio×Dice
   Ignore label 255 is excluded from both.

5. Evaluate on val split every test_freq epochs
   Save checkpoint (fine.pt) whenever val accuracy improves

6. Final evaluation on test split (printed + saved as prediction images)
```

### Why separate learning rates?

The encoder has already learned useful feature representations in Stage 1.
A large LR would destroy those features before the decoder has a chance to use them.
Using a 10× smaller LR for the encoder lets both parts converge together.

In [ ]:
# ── Stage 2: smoke-test run (2 epochs) ───────────────────────────────────────
# Requires Stage 1 checkpoint moco1.pt to exist (written by the smoke-test above).
# For a real run use: python train_metaldam.py --stage fine

# Check checkpoint exists
ckpt = 'self_UNet_metaldam_Numf/f/moco1.pt'
if not os.path.exists(ckpt):
    print(f'Checkpoint not found: {ckpt}')
    print('Run the Stage 1 cell above first.')
else:
    print(f'Checkpoint found: {ckpt}  ({os.path.getsize(ckpt)/1e6:.0f} MB)')
    print('Running 2 finetuning epochs (smoke test) ...')
    print('-' * 60)

    config_metaldam.run(
        method='fine',
        tt='metaldam',
        data_dir=METALDAM_PATCHES,
        load_moco_ep='1',    # load the epoch-1 checkpoint from smoke test
        fine_max_epoch=2,
        env='0',
    )

---
## 8 · Evaluation — predicted vs ground-truth masks

After finetuning, we load the best checkpoint and visualise predictions
on held-out test patches side-by-side with ground truth.

In [ ]:
# ── Load the best finetuned model and run inference ───────────────────────────
from model import UNet
from torch.utils.data import DataLoader

fine_ckpt = 'fine_UNet_metaldam_Numf/f/fine.pt'

if not os.path.exists(fine_ckpt):
    print(f'Finetuned checkpoint not found: {fine_ckpt}')
    print('Run Stage 2 first (or run train_metaldam.py --stage fine for the full run).')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model = UNet(drop=0, channel=[32, 64, 128, 256],
                 IncNorm=['BN','BN'], DownNorm=['BN','BN'], UpNorm=['BN','LN'])
    ckpt_data = torch.load(fine_ckpt, map_location='cpu')
    model.load_state_dict(ckpt_data['finetune'])
    model = model.to(device).eval()
    print(f'Loaded checkpoint from epoch {ckpt_data["epoch"]} (acc={ckpt_data["acc"]:.4f})')

    test_ds = MetalDAMDataset(METALDAM_PATCHES, split='test')
    n_show  = 4
    indices = random.sample(range(len(test_ds)), n_show)

    fig, axes = plt.subplots(3, n_show, figsize=(4*n_show, 10))
    fig.suptitle('Finetuned PSCL — test split predictions', fontsize=12)
    row_labels = ['SEM image', 'Ground truth', 'Prediction']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=11)

    with torch.no_grad():
        for col, idx in enumerate(indices):
            img_t, gt_t, fname = test_ds[idx]
            inp = img_t.unsqueeze(0).to(device)              # (1,3,H,W)
            pred_logits = model(inp)                          # (1,4,H,W)
            pred_cls = torch.softmax(pred_logits, dim=1).cpu().numpy()[0]
            pred_cls = np.argmax(pred_cls, axis=0).astype(np.uint8)  # (H,W)

            # ground truth from one-hot
            gt_np  = gt_t.numpy()
            gt_cls = np.argmax(gt_np, axis=0).astype(np.uint8)
            gt_cls[gt_np.sum(axis=0) == 0] = 255

            # pixel accuracy (ignoring 255)
            valid = (gt_cls != 255)
            acc   = np.mean(pred_cls[valid] == gt_cls[valid]) if valid.any() else 0

            axes[0, col].imshow(unnormalise(img_t))
            axes[0, col].set_title(f'{fname[:18]}', fontsize=7)
            axes[0, col].axis('off')

            axes[1, col].imshow(colorise_pscl_mask(gt_cls))
            axes[1, col].axis('off')

            axes[2, col].imshow(colorise_pscl_mask(pred_cls))
            axes[2, col].set_title(f'acc={acc:.3f}', fontsize=9)
            axes[2, col].axis('off')

    fig.legend(handles=leg2, loc='lower center', ncol=5, fontsize=9, frameon=False)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

In [ ]:
# ── Parse and plot finetuning loss + val metrics ──────────────────────────────
fine_log = 'fine_UNet_metaldam_Numf/f/log_fine.txt'

fine_epochs, fine_loss, val_acc, val_miou = [], [], [], []

if os.path.exists(fine_log):
    with open(fine_log) as f:
        for line in f:
            m = re.search(r'Epoch \[(\d+)/\d+\] Loss ([\d.]+)', line)
            if m:
                fine_epochs.append(int(m.group(1)))
                fine_loss.append(float(m.group(2)))
            m2 = re.search(r'\[val\] ACC ([\d.]+).*mIoU ([\d.]+)', line)
            if m2:
                val_acc.append(float(m2.group(1)))
                val_miou.append(float(m2.group(2)))

if fine_epochs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Stage 2 — Finetuning curves', fontsize=12)

    axes[0].plot(fine_epochs, fine_loss, 'b-o', markersize=3)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training loss (BCE + Dice)')
    axes[0].grid(alpha=0.3)

    if val_acc:
        val_x = [fine_epochs[i*10] for i in range(len(val_acc)) if i*10 < len(fine_epochs)]
        axes[1].plot(range(len(val_acc)), val_acc,  'g-o', markersize=4, label='Val Accuracy')
        axes[1].plot(range(len(val_miou)), val_miou, 'r-o', markersize=4, label='Val mIoU')
        axes[1].set_xlabel('Evaluation checkpoint'); axes[1].set_ylabel('Score')
        axes[1].set_title('Validation metrics')
        axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print(f'Finetuning log not found at {fine_log}')

---
## 9 · Summary

### What this project does, end to end

```
MetalDam/data/patches/
  ├── train/  (788 patches, 256×256 grayscale PNG + uint8 mask)        
  ├── val/    (168 patches)                                             
  └── test/   (192 patches)                                            
        │
        │  data_metaldam.py  ← resize, repeat channel, remap classes, one-hot
        ↓
  Stage 1: SelfSupervised_MetalDAM()   (finetune.py)
    Contrastive pretraining — encoder learns steel microstructure features
    Output: self_UNet_metaldam_Numf/f/moco200.pt
        │
        │  load_moco()   ← strips projection head, keeps encoder weights
        ↓
  Stage 2: Finetune_MetalDAM()         (finetune.py)
    Supervised segmentation — decoder learns to map features to 4 classes
    Output: fine_UNet_metaldam_Numf/f/fine.pt
        │
        ↓
  Metrics on test split:
    mIoU, per-class IoU (Austenite / Matrix / MA / Precipitate), pixel accuracy
```

### Key design decisions

| Decision | Why |
|----------|-----|
| Background → ignore label | Avoids architecture change; Background is not a real phase |
| Mask resize with INTER_NEAREST | Other modes create phantom class indices at boundaries |
| Val split as labeled guidance | Separate from unlabeled pool — no data leakage |
| ID bucketed into 4 groups | PSCL hardcodes `np.eye(6)`; 4 groups fit within 0–5 |
| Encoder LR 10× smaller | Preserves pretrained features while decoder trains from scratch |
| Class weights `[1,1,5,5]` | Compensates for severe class imbalance (MA and Precipitate are rare) |

### Next steps

- Run the full 200-epoch pretraining: `python train_metaldam.py --stage self`
- Run finetuning: `python train_metaldam.py --stage fine`
- Compare against Path B (supervised baseline): modify `fine_max_epoch` and skip Stage 1
- Feed `fine.pt` predictions into `MetalDam/src/analysis/geometric_features.py` for grain analysis